## Example 1: Basic HTTP Caching with FileCache

`FileCache` needs the optional `filelock` package, which a plain `pip install cachecontrol`
does not install. Install the package with its `filecache` extra to run this notebook:
`pip install "cachecontrol[filecache]"`.

In [ ]:
import requests
from cachecontrol import CacheControl
from cachecontrol.caches import FileCache
import tempfile
import os
import shutil
import importlib.util

print("Example 1: Basic HTTP Caching with FileCache")

# FileCache imports filelock lazily; without it the first FileCache(...) call raises ImportError.
assert importlib.util.find_spec('filelock'), 'FileCache needs filelock: pip install "cachecontrol[filecache]"'

try:
    # Create a temporary directory for cache
    cache_dir = tempfile.mkdtemp()
    print(f"✓ Cache directory created: {cache_dir}")
    
    # Set up session with file-based caching
    session = requests.Session()
    cached_session = CacheControl(session, cache=FileCache(cache_dir))
    
    print(f"✓ CacheControl session created with FileCache")
    print(f"Session adapters: {len(cached_session.adapters)}")
    
    # Test the cache object
    cache = FileCache(cache_dir)
    print(f"✓ Cache backend: {type(cache).__name__}")
    print(f"Cache directory exists: {os.path.exists(cache_dir)}")
    
    # Clean up
    shutil.rmtree(cache_dir, ignore_errors=True)
    print("✓ Cache directory cleaned up")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cachecontrol example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
import requests
from cachecontrol import CacheControl

print("Example 2: Memory Cache and Session Usage")

try:
    # Create cached session with in-memory cache (default)
    session = requests.Session()
    cached_session = CacheControl(session)
    
    print(f"✓ CacheControl session created with default cache")
    print(f"Base session type: {type(session).__name__}")
    print(f"Cached session type: {type(cached_session).__name__}")
    
    # Show session attributes
    cached_attrs = len([attr for attr in dir(cached_session) if not attr.startswith('_')])
    session_attrs = len([attr for attr in dir(session) if not attr.startswith('_')])
    print(f"✓ Cached session attributes: {cached_attrs}")
    print(f"Original session attributes: {session_attrs}")
    
    # Demonstrate URL preparation
    url = "https://httpbingo.org/json"
    print(f"✓ Example URL: {url}")
    
    # Test session methods available
    http_methods = ['get', 'post', 'put', 'delete', 'head', 'options', 'patch']
    available_methods = [method for method in http_methods if hasattr(cached_session, method)]
    print(f"✓ Available HTTP methods: {available_methods}")
    
    # Check cache adapter is installed
    adapters = list(cached_session.adapters.keys())
    print(f"✓ Session adapters: {adapters}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cachecontrol example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
import requests
from cachecontrol import CacheControl
from cachecontrol.heuristics import ExpiresAfter
from datetime import timedelta

print("Example 3: Cache Heuristics and Expiration")

try:
    # Create session with expiration heuristic
    session = requests.Session()
    # Cache responses for 60 seconds using heuristic
    heuristic = ExpiresAfter(seconds=60)
    cached_session = CacheControl(session, heuristic=heuristic)
    
    print(f"✓ CacheControl with ExpiresAfter heuristic created")
    print(f"Heuristic type: {type(heuristic).__name__}")
    print(f"Cache expiration: 60 seconds")
    
    # Test heuristic properties
    print(f"✓ Heuristic configured for automatic expiration")
    
    # Demonstrate timedelta usage
    td = timedelta(minutes=5)
    heuristic_5min = ExpiresAfter(minutes=5)
    print(f"✓ Alternative heuristic: 5 minutes ({td.total_seconds()} seconds)")
    
    # Check adapters have heuristic support
    adapters = list(cached_session.adapters.keys())
    print(f"✓ Session adapters with heuristic: {adapters}")
    
    # Test different heuristic configurations
    heuristic_1hour = ExpiresAfter(hours=1)
    heuristic_1day = ExpiresAfter(days=1)
    print(f"✓ 1 hour heuristic: {type(heuristic_1hour).__name__}")
    print(f"✓ 1 day heuristic: {type(heuristic_1day).__name__}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cachecontrol example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
import requests
from cachecontrol import CacheControl
from cachecontrol.adapter import CacheControlAdapter

print("Example 4: Cache Control Headers and Configuration")

try:
    # Create session and examine cache adapter
    session = requests.Session()
    cached_session = CacheControl(session)
    
    print(f"✓ CacheControl session created")
    print(f"Session type: {type(cached_session).__name__}")
    
    # Examine adapters - CacheControl uses HTTPAdapter
    adapters = list(cached_session.adapters.keys())
    print(f"✓ Session adapters: {adapters}")
    
    # Get the cache adapter
    http_adapter = cached_session.adapters.get('http://')
    https_adapter = cached_session.adapters.get('https://')
    
    if http_adapter:
        print(f"✓ HTTP adapter: {type(http_adapter).__name__}")
    if https_adapter:
        print(f"✓ HTTPS adapter: {type(https_adapter).__name__}")
    
    # Show cache configuration options
    print(f"✓ Session supports caching for GET and HEAD requests")
    print(f"Respects Cache-Control, Expires, ETag, Last-Modified headers")
    
    # Test adapter methods
    if http_adapter:
        adapter_methods = [method for method in dir(http_adapter) 
                          if not method.startswith('_') and callable(getattr(http_adapter, method))]
        key_methods = [m for m in adapter_methods if 'cache' in m.lower() or m in ['send', 'build_response']]
        print(f"✓ Key adapter methods: {key_methods}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cachecontrol example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
import requests
from cachecontrol import CacheControl
from cachecontrol.serialize import Serializer
from cachecontrol.controller import CacheController

print("Example 5: Cache Serialization and Key Management")

try:
    # Create session with explicit serializer
    session = requests.Session()
    serializer = Serializer()
    cached_session = CacheControl(session, serializer=serializer)
    
    print(f"✓ CacheControl with custom serializer created")
    print(f"Serializer type: {type(serializer).__name__}")
    
    # Test serializer methods
    serializer_methods = [method for method in dir(serializer) 
                         if not method.startswith('_')]
    print(f"✓ Serializer methods: {serializer_methods}")
    
    # Create a mock request to show key generation
    mock_request = requests.Request('GET', 'https://httpbingo.org/json')
    prepared = session.prepare_request(mock_request)
    
    print(f"✓ Mock request prepared")
    print(f"Method: {prepared.method}")
    print(f"URL: {prepared.url}")
    
    # Show cache controller functionality
    controller = CacheController(serializer=serializer)
    print(f"✓ Cache controller: {type(controller).__name__}")
    
    # Test controller methods
    controller_methods = [method for method in dir(controller) 
                         if not method.startswith('_') and callable(getattr(controller, method))]
    key_methods = [m for m in controller_methods if m in ['cache_url', 'cached_request', 'update_cached_response']]
    print(f"Controller key methods: {key_methods}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise AssertionError(
        f"cachecontrol example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e